# grahspj: Fairall 9 From the VizieR SED Tool

This notebook queries broadband photometry for `Fairall 9` from the CDS VizieR SED service, maps a supported subset of filters into `grahspj`, and fits the resulting SED.

Workflow:
- query the VizieR SED tool by object name
- keep only filters with transmission curves available to `grahspj`
- build a manual `FitConfig`
- run a MAP fit
- inspect the fitted summary and SED plot


## Prerequisites

This notebook assumes:
- you are running from the `notebooks/` directory of this repository
- `grahspj` dependencies are installed
- internet access is available for the VizieR SED query
- a valid DSPS SSP file is available

By default the notebook looks for `../jaxqsofit/tempdata.h5` as a sibling checkout. Adjust that path if your SSP file lives elsewhere.

Notes:
- the VizieR SED service returns many heterogeneous measurements; this notebook keeps only a conservative subset of filters that already have known transmission curves
- when multiple measurements map onto the same filter, the notebook keeps the entry with the smallest fractional flux uncertainty
- the redshift is fixed to a literature value for Fairall 9 in this example rather than fit as a photo-z

- `fit_method="optax"` and the MAP stage of `fit_method="optax+nuts"` use staged MAP/SVI initialization by default.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

project_root = Path.cwd().resolve().parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from grahspj.config import AGNConfig, FilterSet, FitConfig, GalaxyConfig, InferenceConfig, LikelihoodConfig, Observation, PhotometryData
from grahspj.core import GRAHSPJ
from grahspj.mplstyle import style_path
from utils import query_vizier_sed

plt.style.use(style_path())


In [ ]:
target_name = "Fairall 9"
target_redshift = 0.04587
search_radius_arcsec = 3.0

phot_rows, sed_table, vizier_url = query_vizier_sed(
    target_name,
    radius=search_radius_arcsec,
    host=("vizier.cfa.harvard.edu", 443, True),
    fallback_hosts=[("vizier.u-strasbg.fr", 80, False)],
    timeout=120,
)

print("Queried URL:", vizier_url)
print("Rows returned:", len(sed_table))
print("Columns:", sed_table.colnames)
sed_table[:5]


In [ ]:
print("Supported photometric points kept for fitting:")
for row in phot_rows:
    print(row)


In [ ]:
dsps_ssp_fn = project_root.parent / "jaxqsofit" / "tempdata.h5"
assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"

cfg = FitConfig(
    observation=Observation(
        object_id=target_name,
        redshift=target_redshift,
        fit_redshift=False,
    ),
    photometry=PhotometryData(
        filter_names=[row["grahsp_filter"] for row in phot_rows],
        fluxes=[row["flux_mjy"] for row in phot_rows],
        errors=[row["err_mjy"] for row in phot_rows],
        is_upper_limit=[False] * len(phot_rows),
    ),
    filters=FilterSet(
        speclite_names={row["grahsp_filter"]: row["speclite_name"] for row in phot_rows},
        use_grahsp_database=False,
    ),
    galaxy=GalaxyConfig(dsps_ssp_fn=str(dsps_ssp_fn)),
    agn=AGNConfig(agn_type=1),
    likelihood=LikelihoodConfig(),
    inference=InferenceConfig(map_steps=400, learning_rate=5e-3),
    prior_config={
        "log_stellar_mass": {"loc": 10.5, "scale": 1.0},
        "fracAGN_5100": {"loc": 0.8, "scale": 0.15},
        "ebv_gal": {"scale": 0.15},
        "ebv_agn": {"scale": 0.15},
    },
)

print("Configured filters:", cfg.photometry.filter_names)
print("Configured redshift:", cfg.observation.redshift)


In [ ]:
cfg.photometry.psf_fwhm_arcsec = [row["psf_fwhm_arcsec"] for row in phot_rows]
cfg.likelihood.use_host_capture_model = True

# Override default psf_fwhm_arcsec with values if you need to
# cfg.photometry.aperture_diameter_arcsec = [...]


### Host-Capture Model

When `use_host_capture_model=True`, the fit no longer assumes that every band contains the same fraction of host-galaxy light. Instead, the observed photometry is modeled as

$$F_{\mathrm{model},i} = F_{\mathrm{AGN},i} + \eta_i F_{\mathrm{host},i},$$

where $\eta_i$ is a band-dependent host-capture fraction. The AGN is treated as unresolved, while the host contribution is allowed to increase with the effective spatial scale of the measurement.

The effective spatial scale is taken from `aperture_diameter_arcsec` when provided, otherwise from `psf_fwhm_arcsec`. The model fits a shared monotonic curve

$$\eta_i = \sigma\left[s\,\left(\log R_i - \log R_0\right)\right],$$

with fitted turnover scale $R_0$ and slope $s$. Small-PSF or profile-fit measurements therefore capture less host light, while larger apertures or lower-resolution bands asymptotically approach the full host flux.

The likelihood is otherwise unchanged: the corrected model fluxes are compared to the photometry with the same Student-t likelihood, observational errors, intrinsic scatter, systematics term, and AGN variability term used elsewhere in `grahspj`.


In [ ]:
fitter = GRAHSPJ(cfg)
# MAP/SVI initialization is staged by default.
fit_result = fitter.fit(
    fit_method="optax+nuts",
    prior_config=cfg.prior_config,
    dsps_ssp_fn=cfg.galaxy.dsps_ssp_fn,
    optax_steps=cfg.inference.map_steps,
    optax_lr=cfg.inference.learning_rate,
    nuts_warmup=100,
    nuts_samples=50,
    plot_fig=False,
    save_fig=False,
    save_result=False,
    progress_bar=True,
)


In [ ]:
pred = fitter.predict()
fig = fitter.plot_sed(show=True)


In [ ]:
model_flux = np.asarray(pred["pred_fluxes"][0], dtype=float)
host_flux = np.asarray(pred["host_fluxes"][0], dtype=float)
dust_flux = np.asarray(pred["dust_fluxes"][0], dtype=float)
agn_flux = np.asarray(pred["agn_fluxes"][0], dtype=float)
phot_wave = np.asarray([flt.effective_wavelength for flt in fitter.context.filters], dtype=float)

print("filter, eff_wave_A, obs_mJy, err_mJy, model_mJy, host_mJy, dust_mJy, agn_mJy")
for name, wave, obs, err, model, host, dust, agn in zip(
    cfg.photometry.filter_names,
    phot_wave,
    cfg.photometry.fluxes,
    cfg.photometry.errors,
    model_flux,
    host_flux,
    dust_flux,
    agn_flux,
):
    print((name, wave, obs, err, model, host, dust, agn))


## Notes

- If the VizieR SED service returns additional supported filters for your environment, they will be picked up automatically by the mapping cell.
- To fit redshift as well, set `cfg.observation.fit_redshift = True` and provide `cfg.observation.redshift_err`.
- For a fuller posterior exploration, keep `fit_method="optax+nuts"` and increase the warmup/sample counts. For a quick sanity check, switch temporarily to `fit_method="optax"`.
